In [1]:
from pyspark.sql import functions as F

df_date = (
    spark.sql("""
        SELECT explode(
            sequence(
                to_date('2025-10-01'),
                to_date('2026-09-30'),
                interval 1 day
            )
        ) AS date
    """)
)

df_date = (
    df_date
    .withColumn("year", F.year("date"))
    .withColumn("month_number", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("quarter", F.concat(F.lit("Q"), F.quarter("date")))
    .withColumn("week_number", F.weekofyear("date"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn(
        "week_start_date",
        F.date_sub(F.col("date"), F.dayofweek("date") - 2)
    )
    .withColumn(
        "week_end_date",
        F.date_add(F.col("date"), 8 - F.dayofweek("date"))
    )
    .withColumn(
        "year_month",
        F.date_format("date", "yyyy-MM")
    )
)

print("Rows:", df_date.count())
print("Columns:", len(df_date.columns))

display(df_date)

StatementMeta(, a4638858-44f7-46af-a5a6-d24bce8ced6b, 3, Finished, Available, Finished, False)

Rows: 365
Columns: 11


SynapseWidget(Synapse.DataFrame, e4a0505e-cbe8-4327-ae11-c9589461f503)

In [2]:
id="e7k3q1"
print("Minimum date:", df_date.agg(F.min("date")).first()[0])
print("Maximum date:", df_date.agg(F.max("date")).first()[0])

duplicate_dates = (
    df_date
    .groupBy("date")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate dates:", duplicate_dates.count())

StatementMeta(, a4638858-44f7-46af-a5a6-d24bce8ced6b, 4, Finished, Available, Finished, False)

Minimum date: 2025-10-01
Maximum date: 2026-09-30
Duplicate dates: 0


In [3]:
id="q2n8vx"
(
    df_date.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dim_date")
)

StatementMeta(, a4638858-44f7-46af-a5a6-d24bce8ced6b, 5, Finished, Available, Finished, False)

In [4]:
id="m4x7cp"
df_silver_date = spark.read.table("dim_date")

print("Silver dim_date rows:", df_silver_date.count())
print("Silver dim_date columns:", len(df_silver_date.columns))

display(df_silver_date.orderBy("date").limit(10))

StatementMeta(, a4638858-44f7-46af-a5a6-d24bce8ced6b, 6, Finished, Available, Finished, False)

Silver dim_date rows: 365
Silver dim_date columns: 11


SynapseWidget(Synapse.DataFrame, 5a6576ad-94d6-478c-9bfa-36ce145606d9)